# Pipeline hoàn chỉnh — 2 GPU DDP (Kaggle)

Pipeline PyTorch DDP trên **Kaggle** — `vit_large_patch16_224` + CIFAR-100.

| Bước | Run | Global Batch | LR |
|------|-----|--------------|-----|
| 1 | baseline | 32 (16×2) | 1e-4 |
| 2 | lr_scaled | 32 (16×2) | 2e-4 |
| 3 | no_lr_scale | 32 (16×2) | 1e-4 (= baseline) |

**Cấu trúc notebook:**
| Cell | Mô tả |
|------|-------|
| 1 | Khởi tạo môi trường, thư mục, kiểm tra GPU |
| 2 | Ghi `ddp_utils.py` + import |
| 3 | Cấu hình thực nghiệm (`get_pipeline_configs`) |
| 4 | Huấn luyện (`run_training`) → `/kaggle/working/results` |
| 5 | Tổng hợp CSV + gom toàn bộ → `/kaggle/working/results_GOM_VE` |
| 6 | Cẩm nang Download → Google Drive |

Notebook đối chiếu: `pipeline_1gpu.ipynb`

## Cấu hình duy nhất khác biệt giữa 2 notebook

Chỉ thay đổi `WORLD_SIZE` (hiện tại = **2**).

In [ ]:
# %% CODE CELL — Cấu hình WORLD_SIZE
WORLD_SIZE = 2

## Cell 1 — Khởi tạo môi trường

**[Input: None | Output: Trạng thái hệ thống, GPU khả dụng]**

Thiết lập cứng `WORK_DIR = /kaggle/working`, tạo các thư mục con, kiểm tra GPU.

In [ ]:
# %% CODE CELL — Cell 1: Khởi tạo môi trường
# [Input: None | Output: Trạng thái hệ thống, GPU khả dụng]

!pip install -q timm

import sys
from pathlib import Path

import torch

# Kaggle: cố định đường dẫn — KHÔNG auto-detect /content
WORK_DIR = Path("/kaggle/working")
RESULTS_DIR = WORK_DIR / "results"
DOWNLOAD_DIR = WORK_DIR / "results_GOM_VE"
DATA_DIR = WORK_DIR / "data"

for directory in (WORK_DIR, RESULTS_DIR, DOWNLOAD_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()

print("=" * 72)
print("CELL 1 — KHỞI TẠO MÔI TRƯỜNG KAGGLE")
print("=" * 72)
print(f"WORK_DIR     = {WORK_DIR.resolve()}")
print(f"RESULTS_DIR  = {RESULTS_DIR.resolve()}")
print(f"DOWNLOAD_DIR = {DOWNLOAD_DIR.resolve()}")
print(f"DATA_DIR     = {DATA_DIR.resolve()}")
print(f"WORLD_SIZE   = {WORLD_SIZE}")
print(f"GPU khả dụng = {NUM_GPUS}")
print(f"PyTorch      = {torch.__version__}")
print(f"CUDA avail   = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for gpu_index in range(NUM_GPUS):
        print(f"  GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)}")
print("-" * 72)
print("nvidia-smi -L:")
!nvidia-smi -L
print("-" * 72)
assert NUM_GPUS >= WORLD_SIZE, f"Cần >= {WORLD_SIZE} GPU, hiện có {NUM_GPUS}"
print("✓ Môi trường sẵn sàng.")

## Cell 2 — Ghi file `ddp_utils.py` và import

**[Input: Chuỗi source code nội dung | Output: File `ddp_utils.py` trên đĩa, log xác nhận đã load]**

Bắt buộc cho `mp.spawn`/DDP — worker không thể nằm trong `__main__` của notebook.

In [ ]:
# %% CODE CELL — Cell 2: Ghi ddp_utils.py và importlib.reload
# [Input: Chuỗi source code nội dung | Output: File ddp_utils.py trên đĩa, log xác nhận đã load]

import importlib

_DDP_UTILS_SOURCE = r'''
"""
PyTorch DDP utilities for scaling experiments on Kaggle (2x T4).
Uses mp.spawn — no torchrun required. All paths are fixed to /kaggle/working.
"""

from __future__ import annotations

import json
import os
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import timm
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms

# ---------------------------------------------------------------------------
# Kaggle path constants (hardcoded — no Colab /content auto-detection)
# ---------------------------------------------------------------------------
KAGGLE_WORKING_DIR = "/kaggle/working"
KAGGLE_RESULTS_DIR = "/kaggle/working/results"
KAGGLE_DOWNLOAD_DIR = "/kaggle/working/results_GOM_VE"
KAGGLE_DATA_DIR = "/kaggle/working/data"


def ensure_kaggle_dirs() -> dict[str, Path]:
    """Create standard Kaggle working directories and return them as Path objects."""
    dirs = {
        "work": Path(KAGGLE_WORKING_DIR),
        "results": Path(KAGGLE_RESULTS_DIR),
        "download": Path(KAGGLE_DOWNLOAD_DIR),
        "data": Path(KAGGLE_DATA_DIR),
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def download_and_sync_guide(world_size: int) -> None:
    """
    Print step-by-step instructions for downloading Kaggle outputs
    and syncing them to Google Drive (manual upload — Kaggle has no Drive mount).
    """
    tag = f"{world_size}gpu"
    download_dir = Path(KAGGLE_DOWNLOAD_DIR)
    results_dir = Path(KAGGLE_RESULTS_DIR)

    summary_csv = download_dir / f"pipeline_summary_{tag}.csv"
    files_in_download = sorted(p.name for p in download_dir.iterdir() if p.is_file())
    files_in_results = sorted(p.name for p in results_dir.iterdir() if p.is_file())

    border = "=" * 72
    star = "*" * 72
    print(star)
    print("*" + " " * 16 + "CẨM NANG CỨU HỘ — TẢI KẾT QUẢ KAGGLE" + " " * 16 + "*")
    print(star)
    print()
    print("⚠  BỐI CẢNH QUAN TRỌNG")
    print("   Kaggle KHÔNG hỗ trợ mount Google Drive (google.colab.drive → NotImplementedError).")
    print("   Toàn bộ kết quả đã được GOM vào thư mục:")
    print(f"   → {download_dir.resolve()}")
    print("   Bạn PHẢI Download thủ công trước khi tắt session (dữ liệu /kaggle/working sẽ MẤT).")
    print()
    print(border)
    print("  BƯỚC 1 — REFRESH CỘT OUTPUT TRÊN KAGGLE")
    print(border)
    print("  1. Nhìn sang cột bên PHẢI màn hình Kaggle, bấm tab « Output » (biểu tượng 📁).")
    print("  2. Bấm nút Refresh (↻) để cập nhật danh sách file mới nhất.")
    print("  3. Tìm và mở thư mục: results_GOM_VE/")
    print("  4. Xác nhận thấy các file: .pt, .json, .csv (danh sách bên dưới).")
    print()
    print(border)
    print("  BƯỚC 2 — DOWNLOAD VỀ MÁY TÍNH")
    print(border)
    print("  1. Trong results_GOM_VE/, bấm nút Download (⬇) trên từng file HOẶC tải cả thư mục.")
    print(f"  2. File CSV tóm tắt chính: pipeline_summary_{tag}.csv")
    print("  3. Checkpoint mô hình: pipeline_*_model.pt")
    print("  4. Log huấn luyện: pipeline_*_metrics.json, pipeline_*_history.json")
    print()
    print(border)
    print("  BƯỚC 3 — UPLOAD LÊN GOOGLE DRIVE")
    print(border)
    print("  1. Mở https://drive.google.com")
    print("  2. Tạo thư mục: DDP_Experiments/")
    print(f"  3. Tạo subfolder: DDP_Experiments/{tag}/")
    print("  4. Kéo-thả TOÀN BỘ file đã tải từ results_GOM_VE/ vào subfolder đó.")
    print("  5. KHÔNG đổi tên file — giữ nguyên để đối chiếu giữa các lần chạy.")
    print()
    print(border)
    print("  BƯỚC 4 — CHIA SẺ GIỮA 2 NOTEBOOK (tuỳ chọn)")
    print(border)
    print("  Để so sánh Speedup 1 GPU ↔ 2 GPU DDP:")
    print(f"  1. Zip nội dung results_GOM_VE/ sau khi chạy notebook {tag}.")
    print(f"  2. Tạo Kaggle Dataset: ddp-pipeline-{tag}")
    print("  3. Add Dataset làm Input vào notebook còn lại.")
    print()
    print(border)
    print(f"  KIỂM TRA — results_GOM_VE/ ({len(files_in_download)} file)")
    print(border)
    for i, name in enumerate(files_in_download, start=1):
        full = download_dir / name
        marker = " ← CSV tóm tắt" if name == summary_csv.name else ""
        print(f"  {i:2d}. {full.resolve()}{marker}")
    if not files_in_download:
        print("  (Chưa có file — hãy chạy Cell 5 trước!)")
    print()
    print(border)
    print(f"  KIỂM TRA — results/ gốc ({len(files_in_results)} file)")
    print(border)
    for i, name in enumerate(files_in_results, start=1):
        print(f"  {i:2d}. {(results_dir / name).resolve()}")
    if not files_in_results:
        print("  (Chưa có file — hãy chạy Cell 4 trước!)")
    print()
    print(star)
    print("*" + " " * 10 + "NHỚ: Download results_GOM_VE TRƯỚC KHI ĐÓNG NOTEBOOK!" + " " * 10 + "*")
    print(star)


def copy_to_download_dir(src: Path, download_dir: Path | None = None) -> Path:
    """Copy a single file into results_GOM_VE for easy Kaggle Output download."""
    dest_root = download_dir or Path(KAGGLE_DOWNLOAD_DIR)
    dest_root.mkdir(parents=True, exist_ok=True)
    dest = dest_root / src.name
    shutil.copy2(src, dest)
    return dest


def sync_all_results_to_download_dir(
    results_dir: Path | str | None = None,
    download_dir: Path | str | None = None,
) -> list[Path]:
    """
    Copy every file from results/ into results_GOM_VE/.
    Returns sorted list of absolute destination paths.
    """
    src_root = Path(results_dir or KAGGLE_RESULTS_DIR)
    dest_root = Path(download_dir or KAGGLE_DOWNLOAD_DIR)
    dest_root.mkdir(parents=True, exist_ok=True)

    if not src_root.exists():
        raise FileNotFoundError(f"Thư mục nguồn không tồn tại: {src_root.resolve()}")

    copied: list[Path] = []
    for src in sorted(src_root.iterdir()):
        if not src.is_file():
            continue
        dest = dest_root / src.name
        shutil.copy2(src, dest)
        copied.append(dest.resolve())

    return copied


@dataclass
class TrainConfig:
    """Training hyperparameters for one experiment run."""

    exp_name: str = "baseline"
    world_size: int = 1
    local_batch_size: int = 16
    learning_rate: float = 1e-4
    epochs: int = 5
    num_workers: int = 2
    model_name: str = "vit_large_patch16_224"
    num_classes: int = 100
    image_size: int = 224
    save_dir: str = KAGGLE_RESULTS_DIR
    seed: int = 42
    use_amp: bool = True
    warmup_epochs: int = 1

    @property
    def global_batch_size(self) -> int:
        return self.local_batch_size * self.world_size


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_ddp(rank: int, world_size: int, port: str = "12355") -> None:
    os.environ.setdefault("MASTER_ADDR", "localhost")
    os.environ.setdefault("MASTER_PORT", port)
    torch.cuda.set_device(rank)
    dist.init_process_group(
        backend="nccl", rank=rank, world_size=world_size, device_id=rank
    )


def cleanup_ddp() -> None:
    if dist.is_initialized():
        dist.destroy_process_group()


def build_transforms(image_size: int, train: bool = True):
    if train:
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.5071, 0.4867, 0.4408),
                    std=(0.2675, 0.2565, 0.2761),
                ),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.5071, 0.4867, 0.4408),
                std=(0.2675, 0.2565, 0.2761),
            ),
        ]
    )


def build_dataloaders(
    config: TrainConfig,
    rank: int,
    world_size: int,
    data_root: str = KAGGLE_DATA_DIR,
):
    train_ds = datasets.CIFAR100(
        root=data_root,
        train=True,
        download=True,
        transform=build_transforms(config.image_size, train=True),
    )
    val_ds = datasets.CIFAR100(
        root=data_root,
        train=False,
        download=True,
        transform=build_transforms(config.image_size, train=False),
    )

    train_sampler = (
        DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
        if world_size > 1
        else None
    )
    val_sampler = (
        DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)
        if world_size > 1
        else None
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config.local_batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=config.num_workers,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.local_batch_size,
        shuffle=False,
        sampler=val_sampler,
        num_workers=config.num_workers,
        pin_memory=True,
    )
    return train_loader, val_loader, train_sampler


def build_model(config: TrainConfig, device: torch.device) -> nn.Module:
    model = timm.create_model(
        config.model_name,
        pretrained=False,
        num_classes=config.num_classes,
    )
    return model.to(device)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[float, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast("cuda", enabled=True):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += images.size(0)

    if total == 0:
        return 0.0, 0.0
    return total_loss / total, correct / total


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    epoch: int,
    use_amp: bool,
) -> float:
    model.train()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / max(total, 1)


def _train_worker(rank: int, world_size: int, config: TrainConfig, shared: dict) -> None:
    set_seed(config.seed + rank)
    is_ddp = world_size > 1

    if is_ddp:
        setup_ddp(rank, world_size)
    device = torch.device(f"cuda:{rank}")

    train_loader, val_loader, train_sampler = build_dataloaders(
        config, rank, world_size
    )
    model = build_model(config, device)

    if is_ddp:
        model = DDP(model, device_ids=[rank], output_device=rank)

    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.05)
    scaler = GradScaler("cuda", enabled=config.use_amp)

    history: dict[str, list[float]] = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    if is_ddp:
        dist.barrier()
    start = time.perf_counter()

    for epoch in range(config.epochs):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler, device, epoch, config.use_amp
        )
        val_loss, val_acc = evaluate(model, val_loader, device)
        epoch_time = time.perf_counter() - epoch_start

        if rank == 0:
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["epoch_time_sec"].append(epoch_time)
            print(
                f"[{config.exp_name}] epoch {epoch + 1}/{config.epochs} "
                f"loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
                f"time={epoch_time:.1f}s"
            )

        if is_ddp:
            dist.barrier()

    total_time = time.perf_counter() - start

    if rank == 0:
        save_path = Path(config.save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

        state_dict = model.module.state_dict() if is_ddp else model.state_dict()
        ckpt_path = save_path / f"{config.exp_name}_model.pt"
        torch.save(
            {
                "model_state_dict": state_dict,
                "config": asdict(config),
                "history": history,
            },
            ckpt_path,
        )

        result = {
            "exp_name": config.exp_name,
            "world_size": world_size,
            "local_batch_size": config.local_batch_size,
            "global_batch_size": config.global_batch_size,
            "learning_rate": config.learning_rate,
            "epochs": config.epochs,
            "total_time_sec": total_time,
            "avg_epoch_time_sec": sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"]),
            "final_val_acc": history["val_acc"][-1] if history["val_acc"] else 0.0,
            "best_val_acc": max(history["val_acc"]) if history["val_acc"] else 0.0,
            "history": history,
            "checkpoint": str(ckpt_path),
        }
        shared["result"] = result

        metrics_path = save_path / f"{config.exp_name}_metrics.json"
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump({k: v for k, v in result.items() if k != "history"}, f, indent=2)

        history_path = save_path / f"{config.exp_name}_history.json"
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)

        print(f"  Saved checkpoint : {ckpt_path}")
        print(f"  Saved metrics    : {metrics_path}")
        print(f"  Saved history    : {history_path}")

    if is_ddp:
        cleanup_ddp()


def run_training(config: TrainConfig) -> dict[str, Any]:
    """Launch single-GPU or DDP training via mp.spawn (Kaggle-safe)."""
    world_size = config.world_size
    if world_size < 1:
        raise ValueError("world_size must be >= 1")
    if world_size > torch.cuda.device_count():
        raise RuntimeError(
            f"Requested {world_size} GPUs but only {torch.cuda.device_count()} available."
        )

    mp.set_start_method("spawn", force=True)
    manager = mp.Manager()
    shared: dict = manager.dict()

    if world_size == 1:
        _train_worker(0, 1, config, shared)
    else:
        mp.spawn(
            _train_worker,
            args=(world_size, config, shared),
            nprocs=world_size,
            join=True,
        )

    return dict(shared.get("result", {}))


def estimate_amdahl_parallel_fraction(speedup: float, num_gpus: int) -> float:
    """
    Given measured speedup S with N GPUs, solve for parallel fraction P:
        S = 1 / ((1 - P) + P/N)  =>  P = (S - 1) / (S - S/N)
    """
    if num_gpus <= 1 or speedup <= 1.0:
        return 0.0
    denom = speedup - (speedup / num_gpus)
    if abs(denom) < 1e-9:
        return 1.0
    p = (speedup - 1.0) / denom
    return max(0.0, min(1.0, p))


def theoretical_amdahl_speedup(parallel_fraction: float, num_gpus: int) -> float:
    serial = 1.0 - parallel_fraction
    return 1.0 / (serial + parallel_fraction / num_gpus)


def load_shared_result(exp_name: str, search_dirs: list[str | Path]) -> dict[str, Any] | None:
    """Load metrics + history exported by another notebook (via Kaggle Dataset)."""
    for base in search_dirs:
        base = Path(base)
        metrics_path = base / f"{exp_name}_metrics.json"
        history_path = base / f"{exp_name}_history.json"
        if not metrics_path.exists():
            continue
        with open(metrics_path, encoding="utf-8") as f:
            result = json.load(f)
        if history_path.exists():
            with open(history_path, encoding="utf-8") as f:
                result["history"] = json.load(f)
        return result
    return None


def get_pipeline_configs(
    world_size: int,
    *,
    base_lr: float = 1e-4,
    local_batch: int = 16,
    epochs: int = 5,
    save_dir: str = KAGGLE_RESULTS_DIR,
) -> list[TrainConfig]:
    """
    Cùng một pipeline 3 bước cho cả 1 GPU và 2 GPU DDP.

    Ý nghĩa tương đương giữa hai notebook:
      - baseline      : cấu hình chuẩn của môi trường (global batch = local × world_size)
      - lr_scaled     : global batch gấp đôi baseline-1GPU + LR tuyến tính (×2)
      - no_lr_scale   : cùng global batch với lr_scaled nhưng giữ LR baseline
    """
    tag = f"{world_size}gpu"
    common = dict(epochs=epochs, save_dir=save_dir)

    if world_size == 1:
        return [
            TrainConfig(
                exp_name=f"pipeline_baseline_{tag}",
                world_size=1,
                local_batch_size=local_batch,
                learning_rate=base_lr,
                **common,
            ),
            TrainConfig(
                exp_name=f"pipeline_lr_scaled_{tag}",
                world_size=1,
                local_batch_size=local_batch * 2,
                learning_rate=base_lr * 2,
                **common,
            ),
            TrainConfig(
                exp_name=f"pipeline_no_lr_scale_{tag}",
                world_size=1,
                local_batch_size=local_batch * 2,
                learning_rate=base_lr,
                **common,
            ),
        ]

    return [
        TrainConfig(
            exp_name=f"pipeline_baseline_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr,
            **common,
        ),
        TrainConfig(
            exp_name=f"pipeline_lr_scaled_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr * 2,
            **common,
        ),
        TrainConfig(
            exp_name=f"pipeline_no_lr_scale_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr,
            **common,
        ),
    ]
'''

_utils_path = WORK_DIR / "ddp_utils.py"
_utils_path.write_text(_DDP_UTILS_SOURCE.strip() + "\n", encoding="utf-8")

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

if "ddp_utils" in sys.modules:
    importlib.reload(sys.modules["ddp_utils"])

import ddp_utils
from ddp_utils import (
    download_and_sync_guide,
    get_pipeline_configs,
    run_training,
    sync_all_results_to_download_dir,
)

print("=" * 72)
print("CELL 2 — GHI & LOAD ddp_utils.py")
print("=" * 72)
print(f"File đã ghi : {_utils_path.resolve()}")
print(f"Kích thước  : {_utils_path.stat().st_size:,} bytes")
print(f"Module path : {ddp_utils.__file__}")
print("✓ ddp_utils sẵn sàng cho mp.spawn / DDP.")

## Cell 3 — Khởi tạo cấu hình thực nghiệm

**[Input: Tham số Hyperparameters | Output: Log chi tiết cấu hình (LR, Batch size) từng lượt chạy]**

In [ ]:
# %% CODE CELL — Cell 3: Khởi tạo cấu hình thực nghiệm
# [Input: Tham số Hyperparameters | Output: Log chi tiết cấu hình (LR, Batch size) từng lượt chạy]

from pathlib import Path

BASE_LR = 1e-4
LOCAL_BATCH = 16
EPOCHS = 5
RUN_KEYS = ["baseline", "lr_scaled", "no_lr_scale"]

configs = get_pipeline_configs(
    WORLD_SIZE,
    base_lr=BASE_LR,
    local_batch=LOCAL_BATCH,
    epochs=EPOCHS,
    save_dir=str(RESULTS_DIR),
)

print("=" * 72)
print("CELL 3 — CẤU HÌNH THỰC NGHIỆM")
print("=" * 72)
print("Model        : vit_large_patch16_224 (timm)")
print(f"Dataset      : CIFAR-100 → {DATA_DIR.resolve()}")
print(f"Save dir     : {RESULTS_DIR.resolve()}")
print(f"WORLD_SIZE   : {WORLD_SIZE}")
print(f"BASE_LR      : {BASE_LR}")
print(f"LOCAL_BATCH  : {LOCAL_BATCH}")
print(f"EPOCHS       : {EPOCHS}")
print("-" * 72)

for run_key, cfg in zip(RUN_KEYS, configs):
    print(f"[{run_key}]")
    print(f"  exp_name          : {cfg.exp_name}")
    print(f"  world_size        : {cfg.world_size}")
    print(f"  local_batch_size  : {cfg.local_batch_size}")
    print(f"  global_batch_size : {cfg.global_batch_size}")
    print(f"  learning_rate     : {cfg.learning_rate}")
    print(f"  epochs            : {cfg.epochs}")
    print(f"  save_dir          : {Path(cfg.save_dir).resolve()}")
    print()

print("✓ Đã khởi tạo 3 cấu hình pipeline.")

## Cell 4 — Vòng lặp huấn luyện chính

**[Input: Dataset CIFAR-100, Configs | Output: Tiến trình train qua từng epoch, Best Acc, đường dẫn tuyệt đối .pt/.json]**

Mỗi lượt chạy lưu checkpoint, metrics, history vào `/kaggle/working/results`.

In [ ]:
# %% CODE CELL — Cell 4: Vòng lặp huấn luyện chính
# [Input: Dataset CIFAR-100, Configs | Output: Tiến trình train, Best Acc, đường dẫn tuyệt đối .pt/.json]

results = {}

print("=" * 72)
print("CELL 4 — HUẤN LUYỆN PIPELINE")
print("=" * 72)
print(f"Kết quả sẽ lưu tại: {RESULTS_DIR.resolve()}")
print("-" * 72)

for run_key, cfg in zip(RUN_KEYS, configs):
    if run_key == "no_lr_scale" and WORLD_SIZE > 1:
        results[run_key] = results["baseline"]
        print(f"\n[{run_key}] Cùng cấu hình baseline trên {WORLD_SIZE} GPU — bỏ qua train lặp.")
        continue

    print("\n" + "=" * 60)
    print(f"RUN: {run_key}")
    print(f"  world_size={cfg.world_size} | local_bs={cfg.local_batch_size} | lr={cfg.learning_rate}")
    print(f"  global_bs={cfg.global_batch_size} | epochs={cfg.epochs}")
    print("=" * 60)

    results[run_key] = run_training(cfg)

    exp_name = cfg.exp_name
    ckpt_path = RESULTS_DIR / f"{exp_name}_model.pt"
    metrics_path = RESULTS_DIR / f"{exp_name}_metrics.json"
    history_path = RESULTS_DIR / f"{exp_name}_history.json"

    print("-" * 60)
    print(f"KẾT QUẢ [{run_key}]")
    print(f"  Best Val Acc     : {results[run_key]['best_val_acc']:.4f}")
    print(f"  Final Val Acc    : {results[run_key]['final_val_acc']:.4f}")
    print(f"  Total Time (s)   : {results[run_key]['total_time_sec']:.1f}")
    print(f"  Checkpoint (.pt) : {ckpt_path.resolve()}")
    print(f"  Metrics (.json)  : {metrics_path.resolve()}")
    print(f"  History (.json)  : {history_path.resolve()}")
    print(f"  ✓ checkpoint tồn tại : {ckpt_path.exists()}")
    print(f"  ✓ metrics tồn tại    : {metrics_path.exists()}")
    print(f"  ✓ history tồn tại    : {history_path.exists()}")

print("\n" + "=" * 72)
print("✓ Hoàn tất huấn luyện.")
print(f"  Thư mục artifact: {RESULTS_DIR.resolve()}")
print("=" * 72)

## Cell 5 — Tổng hợp kết quả & gom file download

**[Input: Dictionary kết quả từ Cell 4 | Output: DataFrame + danh sách file trong `results_GOM_VE`]**

Tạo `pipeline_summary_{N}gpu.csv`, copy **toàn bộ** file từ `results/` sang `results_GOM_VE/`.

In [ ]:
# %% CODE CELL — Cell 5: Tổng hợp kết quả & gom file download
# [Input: Dictionary kết quả từ Cell 4 | Output: DataFrame + copy toàn bộ sang results_GOM_VE]

import pandas as pd
from IPython.display import display

print("=" * 72)
print("CELL 5 — TỔNG HỢP & GOM FILE DOWNLOAD")
print("=" * 72)

summary_rows = []
for run_key in RUN_KEYS:
    result = results[run_key]
    summary_rows.append({
        "Run": run_key,
        "World Size": result["world_size"],
        "Local Batch": result["local_batch_size"],
        "Global Batch": result["global_batch_size"],
        "LR": result["learning_rate"],
        "Total Time (s)": round(result["total_time_sec"], 2),
        "Best Val Acc": round(result["best_val_acc"], 4),
    })

summary_df = pd.DataFrame(summary_rows)
print("Bảng tổng hợp pipeline:")
display(summary_df)

summary_csv_path = RESULTS_DIR / f"pipeline_summary_{WORLD_SIZE}gpu.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nĐã lưu summary CSV → {summary_csv_path.resolve()}")

print(f"\nĐang copy toàn bộ file từ:\n  {RESULTS_DIR.resolve()}\n→ {DOWNLOAD_DIR.resolve()}")
copied_paths = sync_all_results_to_download_dir(RESULTS_DIR, DOWNLOAD_DIR)

print("-" * 72)
print(f"DANH SÁCH FILE TRONG results_GOM_VE ({len(copied_paths)} file):")
print("-" * 72)
for index, file_path in enumerate(sorted(copied_paths, key=lambda p: p.name), start=1):
    size_kb = file_path.stat().st_size / 1024
    print(f"  {index:2d}. {file_path}  ({size_kb:.1f} KB)")

print("-" * 72)
print("✓ Tất cả file đã gom vào results_GOM_VE.")
print("  → Mở cột Output Kaggle (bên phải) → Refresh → Download.")
print("=" * 72)

## Cell 6 — Cẩm nang hướng dẫn tải file thủ công

**[Input: None | Output: Hướng dẫn Refresh Output → Download → Upload Google Drive `DDP_Experiments/`]**

In [ ]:
# %% CODE CELL — Cell 6: Tự động dò tìm đường dẫn và vẽ đồ thị Amdahl
import ddp_utils
import matplotlib.pyplot as plt
import numpy as np
import json
import os
from pathlib import Path

# =====================================================================
# BƯỚC THẦN THÁNH: TỰ ĐỘNG DÒ ĐƯỜNG DẪN DATASET (Tránh lỗi gạch dưới/gạch ngang của Kaggle)
# =====================================================================
base_input = Path("/kaggle/input")
found_results_dir = None

# Quét toàn bộ thư mục input để tìm folder mang tên 'results' như trong hình image_6345b8.png
for path in base_input.glob("**/results"):
    if path.is_dir():
        found_results_dir = path
        break

if found_results_dir is None:
    print("❌ Lỗi hệ thống: Không tìm thấy thư mục 'results' nào trong vùng Input.")
    print("Các thư mục hiện có trong /kaggle/input là:", os.listdir("/kaggle/input"))
else:
    INPUT_1GPU_DIR = found_results_dir / "1gpu"
    INPUT_2GPU_DIR = found_results_dir / "2gpu"
    print(f"🎯 Đã tìm thấy dataset tại đường dẫn thực tế: {found_results_dir}")

    # Thư mục để lưu kết quả đầu ra mới của cell này
    RESULTS_DIR = Path("/kaggle/working/results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    # Đọc dữ liệu trực tiếp thông qua ddp_utils
    my_baseline = ddp_utils.load_shared_result("pipeline_baseline_1gpu", [INPUT_1GPU_DIR])
    other_baseline = ddp_utils.load_shared_result("pipeline_baseline_2gpu", [INPUT_2GPU_DIR])

    if my_baseline and other_baseline:
        t1 = my_baseline["total_time_sec"]
        t2 = other_baseline["total_time_sec"]

        speedup = t1 / t2
        P_est = ddp_utils.estimate_amdahl_parallel_fraction(speedup, num_gpus=2)

        print(f"Thời gian 1 GPU : {t1:.1f}s")
        print(f"Thời gian 2 GPU : {t2:.1f}s")
        print(f"Speedup: {speedup:.3f}x | P (Amdahl): {P_est:.3f}")

        # Vẽ đồ thị
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].bar(["1 GPU", "2 GPU"], [t1, t2], color=["#4C72B0", "#55A868"])
        axes[0].set_ylabel("Time (s)")
        axes[0].set_title("Baseline training time")

        gpu_range = np.arange(1, 5)
        axes[1].plot(gpu_range, [ddp_utils.theoretical_amdahl_speedup(P_est, n) for n in gpu_range], "o-", label=f"Amdahl P={P_est:.2f}")
        axes[1].plot([1, 2], [1, speedup], "s--", color="red", label=f"Đo được {speedup:.2f}x")
        axes[1].plot(gpu_range, gpu_range, ":", color="gray", label="Linear")
        axes[1].set_xlabel("N GPU")
        axes[1].set_ylabel("Speedup")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        plt.tight_layout()

        # Lưu biểu đồ và báo cáo vào thư mục working mới
        plt.savefig(RESULTS_DIR / "amdahl_analysis.png", dpi=150)
        plt.show()

        report = {"speedup": speedup, "P": P_est, "t_1gpu": t1, "t_2gpu": t2}
        with open(RESULTS_DIR / "amdahl_report.json", "w") as f:
            json.dump(report, f, indent=2)
        print("✓ Đã xuất đồ thị và file report thành công tại /kaggle/working/results!")
    else:
        print("❌ Lỗi: Cấu trúc thư mục đúng nhưng không đọc được nội dung tệp JSON bên trong.")